In [24]:
from tqdm.auto import tqdm
import pandas as pd
from typing import Optional, List, Tuple
import json
import datasets

pd.set_option("display.max_colwidth", None)

In [25]:
from huggingface_hub import notebook_login

notebook_login()

In [26]:
from langchain_community.document_loaders import DirectoryLoader
loader = DirectoryLoader("C:\\Users\\Aiml cse\\Desktop\\ISFCR_Codes\\pdfs")
documents = loader.load()

In [27]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document as LangchainDocument

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    add_start_index=True,
    separators=["\n\n", "\n", ".", " ", ""],
)

docs_processed = []
for doc in documents:
    docs_processed += text_splitter.split_documents([doc])

In [28]:
from huggingface_hub import InferenceClient


repo_id = "mistralai/Mixtral-8x7B-Instruct-v0.1"

llm_client = InferenceClient(
    model=repo_id,
    timeout=120,
)


def call_llm(inference_client: InferenceClient, prompt: str):
    response = inference_client.post(
        json={
            "inputs": prompt,
            "parameters": {"max_new_tokens": 1000},
            "task": "text-generation",
        },
    )
    return json.loads(response.decode())[0]["generated_text"]


call_llm(llm_client, "This is a test context")

'This is a test context for the `@mui/material` library.\n\n## Installation\n\n```sh\nnpm install @mui/material\n```\n\n## Usage\n\n```jsx\nimport React from \'react\';\nimport { Button } from \'@mui/material\';\n\nfunction App() {\n  return (\n    <div className="App">\n      <Button variant="contained" color="primary">\n        Hello World\n      </Button>\n    </div>\n  );\n}\n\nexport default App;\n```\n\n## Documentation\n\n- [Material-UI](https://material-ui.com/)\n- [Material Design](https://material.io/)'

In [29]:
QA_generation_prompt = """
Your task is to write a factoid question and an answer given a context.
Your factoid question should be answerable with a specific, concise piece of factual information from the context.
Your factoid question should be formulated in the same style as questions users could ask in a search engine.
This means that your factoid question MUST NOT mention something like "according to the passage" or "context".

Provide your answer as follows:

Output:::
Factoid question: (your factoid question)
Answer: (your answer to the factoid question)

Now here is the context.

Context: {context}\n
Output:::"""

In [30]:
import random

N_GENERATIONS = 10  # We intentionally generate only 10 QA couples here for cost and time considerations

print(f"Generating {N_GENERATIONS} QA couples...")

outputs = []
for sampled_context in tqdm(random.sample(docs_processed, N_GENERATIONS)):
    # Generate QA couple
    output_QA_couple = call_llm(llm_client, QA_generation_prompt.format(context=sampled_context.page_content))
    try:
        question = output_QA_couple.split("Factoid question: ")[-1].split("Answer: ")[0]
        answer = output_QA_couple.split("Answer: ")[-1]
        assert len(answer) < 300, "Answer is too long"
        outputs.append(
            {
                "context": sampled_context.page_content,
                "question": question,
                "answer": answer,
                "source_doc": sampled_context.metadata["source"],
            }
        )
    except:
        continue

Generating 10 QA couples...


  0%|          | 0/10 [00:00<?, ?it/s]

In [31]:
pd.DataFrame(outputs).to_csv("qa.csv")

In [32]:
display(pd.DataFrame(outputs).head())

,context,question,answer,source_doc
0,"Authors: • M. Sundarramurthi • Adithya Balasubramanyam • Ashok Kumar Patil\n\nAbstract: Over the years, navigation has become an essential part of our daily lives. The emergence of Augmented Reality (AR) technology has created new opportunities for improving the navigation experience. The current work focuses on the development of NavPES, a mobile navigation application designed to provide users with seamless and efficient navigation throughout the PES University campus, both indoors and outdoors. The application makes use of cutting-edge tools such as the ARway software development kit (SDK), Vuforia SDK, and Azure Spatial Anchors to enable accurate indoor localization and navigation using AR. NavPES features an interactive and self-explanatory Graphical User Interface (GUI) for users, allowing them to navigate around the campus with ease. The use of AR technology in navigation applications like NavPES contributes to the development of smart digital campuses.\n\nPESU PESU Center for T\n\nInternet of Things30\n\nXAI for Securing Cyber Physical Systems\n\nConference published in: 2023 Third International Conference on Secure Cyber Computing and Communication (ICSCCC), Jalandhar, India, 2023, pp. 671- 677, doi: 10.1109/ICSCCC58608.2023.10176832 Indexing: Scopus\n\nAuthors: • Amar Prakash Patil • Joshitha Devarakonda • Manognya Singuru • Smriti Tilak • Shruti Jadon","Who are the authors of the paper ""XAI for Securing Cyber Physical Systems""?\n","Amar Prakash Patil, Joshitha Devarakonda, Manognya Singuru, Smriti Tilak, and Shruti Jadon",C:\Users\Aiml cse\Desktop\ISFCR_Codes\pdfs\Research_Vol_2023.pdf
1,". The application is made generic by making the AR system be able to provide an AR interface to any application that can run on a regular PC. The accessibility of the AR system is improved by making it compatible to work on any normal smartphone with a regular camera. There is no necessity for a depth- sensing camera, which is a requirement of popular AR toolkits like",What is the requirement for the AR system to work on a normal smartphone?\n,The AR system does not require a depth-sensing camera to work on a normal smartphone.,C:\Users\Aiml cse\Desktop\ISFCR_Codes\pdfs\Research_Vol_2023.pdf
2,"9\n\nHomomorphic Encryption Approach for String Concatenation\n\nConference published in: 2022 IEEE 4th International Conference on Cybernetics, Cognition and Machine Learning Applications (ICCCMLA), Goa, India, 2022, pp. 267-272 doi: 10.1109/ICCCMLA56841.2022.9989264 Indexing: Scopus\n\nAuthors: • S Rajashree • B Vineetha • Ami B Mehta • Prasad B Honnavalli\n\nAbstract: In a modern environment, a large amount of data is generated from various sources like healthcare, government, banking, and other different sectors. The traditional method for storing information is becoming challenging to handle big data. The solution to storing big data is to store data on third-party service providers. Data protection is the primary concern for storing and accessing data due to the emerging rate of data breaches and adversaries in personal data storage. Intruders manage to access sensitive data despite many efforts to protect sensitive data. Homomorphic encryption is a data protection approach in the cryptographic domain that can perform computations\n\non cipher data without decrypting it. The computation outcome is encrypted; the output will be the same when decrypted as if the operations had been performed on the original plaintext data. This encryption method for privacy-preserving outsourced storage and computation permits the data to be encrypted and outsourced to cloud environments for further processing. This paper proposes string concatenation on encrypted data using the Advanced Encryption Standard algorithm with cipher blockchain mode of operation.\n\n10\n\nCoreMedi: Secure Medical Records Sharing Using Blockchain Technology\n\nConference published in: 2022 International Conference on Data Analytics for 

In [33]:
question_groundedness_critique_prompt = """
You will be given a context and a question.
Your task is to provide a 'total rating' scoring how well one can answer the given question unambiguously with the given context.
Give your answer on a scale of 1 to 5, where 1 means that the question is not answerable at all given the context, and 5 means that the question is clearly and unambiguously answerable with the context.

Provide your answer as follows:

Answer:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here are the question and context.

Question: {question}\n
Context: {context}\n
Answer::: """

question_relevance_critique_prompt = """
You will be given a question.
Your task is to provide a 'total rating' representing how useful this question can be to machine learning developers building NLP applications with the Hugging Face ecosystem.
Give your answer on a scale of 1 to 5, where 1 means that the question is not useful at all, and 5 means that the question is extremely useful.

Provide your answer as follows:

Answer:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here is the question.

Question: {question}\n
Answer::: """

question_standalone_critique_prompt = """
You will be given a question.
Your task is to provide a 'total rating' representing how context-independant this question is.
Give your answer on a scale of 1 to 5, where 1 means that the question depends on additional information to be understood, and 5 means that the question makes sense by itself.
For instance, if the question refers to a particular setting, like 'in the context' or 'in the document', the rating must be 1.
The questions can contain obscure technical nouns or acronyms like Gradio, Hub, Hugging Face or Space and still be a 5: it must simply be clear to an operator with access to documentation what the question is about.

For instance, "What is the name of the checkpoint from which the ViT model is imported?" should receive a 1, since there is an implicit mention of a context, thus the question is not independant from the context.

Provide your answer as follows:

Answer:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here is the question.

Question: {question}\n
Answer::: """

In [34]:
print("Generating critique for each QA couple...")
for output in tqdm(outputs):
    evaluations = {
        "groundedness": call_llm(
            llm_client,
            question_groundedness_critique_prompt.format(context=output["context"], question=output["question"]),
        ),
        "relevance": call_llm(
            llm_client,
            question_relevance_critique_prompt.format(question=output["question"]),
        ),
        "standalone": call_llm(
            llm_client,
            question_standalone_critique_prompt.format(question=output["question"]),
        ),
    }
    try:
        for criterion, evaluation in evaluations.items():
            score, eval = (
                int(evaluation.split("Total rating: ")[-1].strip()),
                evaluation.split("Total rating: ")[-2].split("Evaluation: ")[1],
            )
            output.update(
                {
                    f"{criterion}_score": score,
                    f"{criterion}_eval": eval,
                }
            )
    except Exception as e:
        continue

Generating critique for each QA couple...


  0%|          | 0/10 [00:00<?, ?it/s]

In [35]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

generated_questions = pd.DataFrame.from_dict(outputs)

print("Evaluation dataset before filtering:")
display(
    generated_questions[
        [
            "question",
            "answer",
            "groundedness_score",
            "relevance_score",
            "standalone_score",
        ]
    ]
)
generated_questions = generated_questions.loc[
    (generated_questions["groundedness_score"] >= 4)
    & (generated_questions["relevance_score"] >= 4)
    & (generated_questions["standalone_score"] >= 4)
]
print("============================================")
print("Final evaluation dataset:")
display(
    generated_questions[
        [
            "question",
            "answer",
            "groundedness_score",
            "relevance_score",
            "standalone_score",
        ]
    ]
)

eval_dataset = datasets.Dataset.from_pandas(generated_questions, split="train", preserve_index=False)

Evaluation dataset before filtering:


,question,answer,groundedness_score,relevance_score,standalone_score
0,"Who are the authors of the paper ""XAI for Securing Cyber Physical Systems""?\n","Amar Prakash Patil, Joshitha Devarakonda, Manognya Singuru, Smriti Tilak, and Shruti Jadon",5,1,5
1,What is the requirement for the AR system to work on a normal smartphone?\n,The AR system does not require a depth-sensing camera to work on a normal smartphone.,5,2,5
2,What is the encryption method used for privacy-preserving outsourced storage and computation in the context?\n,The encryption method used for privacy-preserving outsourced storage and computation in the context is homomorphic encryption.,5,1,3
3,What is the responsibility of network administrators in handling large amounts of data?\n,The responsibility of network administrators in handling large amounts of data is to transmit and store the data securely.,3,2,5
4,Which database did the authors implement differential privacy in?\n,The authors implemented differential privacy in the NoSQL database MongoDB.,5,1,2
5,"Which conference was the paper ""An Android-Based Multifactor Authentication for Securing Passive Keyless Access System"" published in?\n","The paper ""An Android-Based Multifactor Authentication for Securing Passive Keyless Access System"" was published in the 2022 IEEE 7th International conference for Convergence in Technology (I2CT).",1,1,5
6,What is the main end goal of cyber-attacks?\n,The main end goal of cyber-attacks is data exfiltration.,2,2,5
7,What is the accuracy of the model developed for emotion detection from text?\n,The model developed for emotion detection from text has an accuracy of 83%.,5,1,4
8,Who are the authors of the paper?\n,"The authors of the paper are V. Harikrishnan, H. S. Sanket, K. S. Sahazeer, Siddarth Vinay Prasad, and B. Honnavalli.",5,1,5
9,What is the accuracy of the Random Forest approach in traditional ML algorithms?\n,The accuracy of the Random Forest approach in traditional ML algorithms is 86%.,2,2,5


Final evaluation dataset:


,question,answer,groundedness_score,relevance_score,standalone_score
